# 09 Counterfactual Explanations

Counterfactuals translate model output into what-if guidance for a high-risk applicant.

## Reproducibility note

These notebooks are lightweight narrative companions to the source-controlled pipeline. The canonical workflow lives in src, generated metrics and figures live under reports, and the final selected model is models/xgboost_public.pkl. The protected attribute SEX is excluded from active model training and retained only for fairness diagnostics.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, Markdown

from src.utils import ROOT_DIR, REPORTS_DIR, MODELS_DIR, load_dataset_auto, load_model
from src.data_preprocessing import TARGET_COL

ROOT = ROOT_DIR
REPORTS = REPORTS_DIR
MODELS = MODELS_DIR
PROCESSED_DATA = ROOT / 'data' / 'processed' / 'uci_taiwan_credit_default_processed.csv'

pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')


def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def show_image_if_exists(path: Path, width: int = 900):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing optional image: {path.relative_to(ROOT) if path.is_absolute() else path}')


def load_project_dataset():
    try:
        frame, source = load_dataset_auto()
        return frame, source
    except Exception as exc:
        if PROCESSED_DATA.exists():
            print(f'UCI loader failed; using processed local fallback. Reason: {exc}')
            return pd.read_csv(PROCESSED_DATA), PROCESSED_DATA
        raise

raw_df, DATA_SOURCE = load_project_dataset()
print('Project root:', ROOT)
print('Data source:', DATA_SOURCE)
print('Rows:', f'{len(raw_df):,}')
print('Columns:', raw_df.shape[1])

cf = load_json(REPORTS / 'explainability_reports' / 'application_model' / 'xgboost_public_counterfactuals.json')
feature_names = cf['feature_names']
original = pd.Series(cf['test_data'][0][0][: len(feature_names)], index=feature_names, name='original')
first_cf = pd.Series(cf['cfs_list'][0][0][: len(feature_names)], index=feature_names, name='counterfactual_1')
changes = pd.DataFrame({'original': original, 'counterfactual': first_cf})
changes = changes[changes['original'].astype(str) != changes['counterfactual'].astype(str)]
display(changes)

Project root: D:\PGDBA\Projects\Credit Default Risk\credit-default-xai
Data source: uci:\default_credit_card
Rows: 30,000
Columns: 38


,original,counterfactual
BillToLimitRatio_1,0.195650,0.195650
BillToLimitRatio_2,0.155100,0.155100
BillToLimitRatio_3,0.034450,0.034450
BillToLimitRatio_4,0.000000,2.100000
AvgBillToLimitRatio,0.064200,0.064200
AvgPaymentToBillRatio,0.037019,0.037019
MaxPaymentDelay,2.000000,1.000000
AvgPaymentAmount,114.833336,114.833333
PaymentToLimitRatio,0.005742,0.005742


Counterfactuals are used as guidance examples only. They are not guarantees of approval and should remain inside a human-review workflow.